In [ ]:
import requests
import pandas as pd
import time

# -----------------------------
# GitHub token
# -----------------------------
GITHUB_TOKEN = ""

headers = {
    "Authorization": f"token {GITHUB_TOKEN}"
}

# -----------------------------
# Repository list
# -----------------------------
repos = [
    "crewaiinc/crewai"
        "liam-hq/liam",
    "airbytehq/airbyte",
    "onlook-dev/onlook",
    "calcom/cal.com",
    "antiwork/flexile",
    "oven-sh/bun",
    "glaredb/glaredb",
    "promptfoo/promptfoo",
    "dotnet/aspire",
    "box/box-ui-elements",
    "microsoft/vscode",
    "antiwork/gumroad",
    "prebid/prebid.js",
    "bionic-gpt/bionic-gpt",
    "julep-ai/julep",
    "goat-sdk/goat",
    "mlflow/mlflow",
    "pdfme/pdfme",
    "novuhq/novu",
    "openhft/chronicle-wire",
    "onekeyhq/app-monorepo",
    "lingodotdev/lingo.dev",
    "antiwork/shortest",
    "appsmithorg/appsmith",
    "remotion-dev/remotion",
    "marimo-team/marimo",
    "near/nearcore",
    "stack-auth/stack-auth",
    "seasonedcc/remix-forms",
    "metacraft-labs/codetracer",
    "yamadashy/repomix",
    "azure/azure-sdk-for-js",
    "elizaos/eliza",
    "mendableai/firecrawl",
    "featureform/enrichmcp",
    "rainbow-me/rainbowkit",
    "tokens-studio/figma-plugin",
    "reown-com/appkit",
    "gofiber/fiber",
    "dotnet/runtime",
    "agentops-ai/agentops",
    "significant-gravitas/autogpt",
    "microsoft/testfx",
    "wix/react-native-ui-lib",
    "azure/azure-sdk-for-python",
    "go-vikunja/vikunja",
    "celestiaorg/celestia-core",
    "jina-ai/node-deepresearch",
    "dotnet/aspnetcore"
]

# -----------------------------
# Years
# -----------------------------
years = [2023, 2024, 2025]

# -----------------------------
# Count review comments
# -----------------------------
def get_review_comment_count(owner, repo, pr_number):

    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/comments"

    params = {"per_page": 1}

    response = requests.get(url, headers=headers, params=params)

    if "Link" not in response.headers:
        return len(response.json())

    link_header = response.headers["Link"]

    if 'rel="last"' in link_header:
        last_page = link_header.split("page=")[-1].split(">")[0]
        return int(last_page)

    return 1


# -----------------------------
# Fetch PRs
# -----------------------------
def fetch_prs(owner, repo, year):

    prs = []
    page = 1

    while True:

        query = f"repo:{owner}/{repo} is:pr is:closed created:{year}-01-01..{year}-12-31"

        url = "https://api.github.com/search/issues"

        params = {
            "q": query,
            "per_page": 100,
            "page": page
        }

        response = requests.get(url, headers=headers, params=params)

        data = response.json()

        items = data.get("items", [])

        if not items:
            break

        prs.extend(items)

        page += 1

        if page > 5:   # max 500 PR per year
            break

    return prs


# -----------------------------
# Main crawler
# -----------------------------
dataset = []
counter = 0

for repo_full in repos:

    owner, repo = repo_full.split("/")

    print("Processing repository:", repo_full)

    for year in years:

        print("  Year:", year)

        prs = fetch_prs(owner, repo, year)

        print("  PRs downloaded:", len(prs))

        for pr in prs:

            pr_number = pr["number"]

            review_count = get_review_comment_count(owner, repo, pr_number)

            dataset.append({
                "repo": repo_full,
                "pr_number": pr_number,
                "title": pr["title"],
                "body": pr["body"],
                "created_at": pr["created_at"],
                "closed_at": pr["closed_at"],
                "user": pr["user"]["login"],
                "review_comments": review_count
            })

            counter += 1

            # save every 50 PRs
            if counter % 50 == 0:
                temp_df = pd.DataFrame(dataset)
                temp_df.to_csv("raw_pr_dataset_partial.csv", index=False)
                print("Saved progress:", counter, "PRs")

            time.sleep(0.2)


# -----------------------------
# Convert to DataFrame
# -----------------------------
df = pd.DataFrame(dataset)

# -----------------------------
# Convert timestamps
# -----------------------------
df["created_at"] = pd.to_datetime(df["created_at"])
df["closed_at"] = pd.to_datetime(df["closed_at"])

# -----------------------------
# Resolution time
# -----------------------------
df["resolution_time_hours"] = (
    df["closed_at"] - df["created_at"]
).dt.total_seconds() / 3600


# -----------------------------
# Save final dataset
# -----------------------------
df.to_csv("raw_pr_dataset.csv", index=False)

print("Dataset saved as raw_pr_dataset.csv")
print("Total PR collected:", len(df))